# 📊 AutoML Engine Comparison on a Real Kaggle Dataset — Adult Census Income

This notebook is the same idea as `automl_engine_comparison.ipynb` (drive Decisera's live `/api/v1/models/train` endpoint once per engine and compare results) but swaps the synthetic benchmark data for a real, well-known Kaggle dataset: the **UCI Adult Census Income** dataset (`uciml/adult-census-income`).

### Why this dataset
- **Binary classification**, ~48.8K rows — big enough for the engines' search budgets to matter, small enough to finish in a few minutes per engine.
- **Realistic mixed schema**: numeric columns (`age`, `hours.per.week`, `capital.gain`, ...) alongside categorical ones (`workclass`, `education`, `occupation`, `marital.status`, `native.country`, ...), so engines actually have to handle encoding/imputation, not just plug in numbers.
- **Missing values** (`?` in several categorical columns) and **class imbalance** (~76% / 24% split on income), which is closer to real production data than a clean synthetic set.
- It's a different domain from the other notebooks in this repo (smart city, telco churn, predictive maintenance), so it also broadens what Decisera has been benchmarked against.

### Flow
1. Download the dataset from Kaggle via `kagglehub`.
2. Light cleaning (normalize `?` to real NaNs, binarize the target) — everything else (imputation, categorical encoding) is left to the backend's own `AutoML.preprocess_fit`, matching what a real Decisera user uploading this CSV as-is would get.
3. Upload the cleaned CSV to Decisera via `/api/v1/datasets/upload`.
4. Kick off one `/api/v1/models/train` run per engine (H2O / FLAML / AutoGluon), same target and time budget.
5. Poll each task to completion and compare `best_model` / `best_score` / leaderboards.

### Requirements
- A running Decisera stack (`docker compose up`) including `backend`, `h2o`, and `autogluon`.
- A Kaggle account with an API token configured for `kagglehub` (or run once interactively to authenticate).
- If a requested engine's service is down, `AutoML.fit()` falls back to the next engine and logs it — check the "Actual Engine" column in the comparison table below if a result looks off.


## 1. Setup


In [ ]:
!pip install -q kagglehub


In [ ]:
import os
import io
import time
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

# Same BACKEND_URL convention used by the other notebooks: defaults to the docker-compose service
# name, override with an env var when running outside Docker (e.g. BACKEND_URL=http://localhost:8000
# if you're on the host machine, not inside the `notebook` container).
BACKEND_URL = os.environ.get("BACKEND_URL", "http://backend:8000")
API_BASE = f"{BACKEND_URL}/api/v1"
print(f"Using Decisera backend at: {API_BASE}")


## 2. Verify the backend is reachable


In [ ]:
resp = requests.get(f"{BACKEND_URL}/health", timeout=5)
resp.raise_for_status()
print("Backend health:", resp.json())


## 3. Download the Adult Census Income dataset from Kaggle


In [ ]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download("uciml/adult-census-income")
print(f"Dataset downloaded to: {dataset_path}")

csv_files = list(Path(dataset_path).glob("*.csv"))
print("Available CSV files:", [f.name for f in csv_files])

raw_df = pd.read_csv(csv_files[0])
print(f"Shape: {raw_df.shape}")
raw_df.head()


## 4. Light cleaning

This dataset encodes missing values as the literal string `"?"` (mostly in `workclass`, `occupation`, and `native.country`) rather than empty cells, so `pandas`/the backend won't recognize them as NaN unless we convert them first. The target column (`income`) is text (`<=50K` / `>50K`, sometimes with a trailing period depending on the file) — we normalize it to a clean binary `0`/`1` so `task_type` resolves to classification unambiguously.

Everything else — imputing the resulting NaNs, encoding the remaining categorical columns — is left to the backend's `AutoML.preprocess_fit`, the same as any real dataset upload.


In [ ]:
clean_df = raw_df.replace("?", np.nan).copy()

target_col = "income"
clean_df[target_col] = (
    clean_df[target_col].astype(str).str.strip().str.rstrip(".").map({"<=50K": 0, ">50K": 1})
)
assert clean_df[target_col].isna().sum() == 0, "Unexpected income value outside {<=50K, >50K}"

print("Missing values per column:")
print(clean_df.isna().sum()[clean_df.isna().sum() > 0])
print(f"\nTarget balance:\n{clean_df[target_col].value_counts(normalize=True)}")


## 5. Upload the cleaned dataset to Decisera


In [ ]:
csv_bytes = clean_df.to_csv(index=False).encode("utf-8")
dataset_name = f"adult_census_income_{int(time.time())}"

upload_resp = requests.post(
    f"{API_BASE}/datasets/upload",
    files={"file": (f"{dataset_name}.csv", io.BytesIO(csv_bytes), "text/csv")},
    data={"name": dataset_name, "description": "UCI Adult Census Income (Kaggle: uciml/adult-census-income) for AutoML engine comparison"},
    timeout=60,
)
upload_resp.raise_for_status()
dataset_id = upload_resp.json()["dataset"]["id"]
print(f"Uploaded dataset '{dataset_name}' -> dataset_id={dataset_id}")


## 6. Kick off one training run per engine

Each config maps directly onto `TrainModelRequest`'s `use_h2o` / `use_flaml` / `use_autogluon` flags (see `backend/api/models.py`). They're mutually exclusive per request so each run is unambiguously attributable to one engine.


In [ ]:
TIME_BUDGET_SECS = 180  # search budget given to each engine; raise for a more thorough (slower) comparison

engine_configs = [
    {
        "label": "H2O",
        "payload": {
            "use_h2o": True,
            "h2o_max_runtime_secs": TIME_BUDGET_SECS,
            "use_flaml": False,
            "use_autogluon": False,
        },
    },
    {
        "label": "FLAML",
        "payload": {
            "use_h2o": False,
            "use_flaml": True,
            "flaml_time_budget_secs": TIME_BUDGET_SECS,
            "use_autogluon": False,
        },
    },
    {
        "label": "AutoGluon",
        "payload": {
            "use_h2o": False,
            "use_flaml": False,
            "use_autogluon": True,
            "autogluon_time_limit": TIME_BUDGET_SECS,
            "autogluon_presets": "medium_quality",
        },
    },
]

task_ids = {}
for cfg in engine_configs:
    train_req = {
        "dataset_id": dataset_id,
        "target_column": target_col,
        "task_type": "classification",
        "test_size": 0.2,
        "experiment_name": f"Adult_Census_Income_Engine_Comparison_{cfg['label']}",
        **cfg["payload"],
    }
    resp = requests.post(f"{API_BASE}/models/train", json=train_req, timeout=30)
    resp.raise_for_status()
    task_id = resp.json()["task_id"]
    task_ids[cfg["label"]] = task_id
    print(f"[{cfg['label']}] training started -> task_id={task_id}")


## 7. Poll each task until it completes (or fails)


In [ ]:
def poll_task(task_id: str, label: str, timeout_secs: int = 900, interval_secs: int = 5) -> dict:
    """Poll /models/tasks/{task_id}/status until status is completed/failed or timeout_secs elapses."""
    deadline = time.time() + timeout_secs
    last_status = None
    while time.time() < deadline:
        resp = requests.get(f"{API_BASE}/models/tasks/{task_id}/status", timeout=10)
        resp.raise_for_status()
        status = resp.json()
        if status.get("status") != last_status:
            print(f"[{label}] status={status.get('status')} progress={status.get('progress')} message={status.get('message')}")
            last_status = status.get("status")
        if status.get("status") in ("completed", "failed"):
            return status
        time.sleep(interval_secs)
    raise TimeoutError(f"[{label}] task {task_id} did not finish within {timeout_secs}s")


engine_statuses = {}
for label, task_id in task_ids.items():
    engine_statuses[label] = poll_task(task_id, label)


## 8. Compare results across engines

`results` on a completed task is the same dict `AutoML.fit()` returns (see `backend/ml/automl.py`): `engine`, `best_model`, `best_score`, `all_results` (per-candidate-model metrics/leaderboard rows), and `task_type`. `Actual Engine` reflects what actually ran the training -- if a requested engine's service was unreachable, this shows the fallback engine instead.


In [ ]:
comparison_rows = []
for label, status in engine_statuses.items():
    if status.get("status") != "completed":
        comparison_rows.append({
            "Requested Engine": label,
            "Actual Engine": "N/A",
            "Best Model": "N/A",
            "Best Score": np.nan,
            "Task Type": "N/A",
            "Model ID": "N/A",
            "Error": status.get("error", status.get("message", "unknown failure")),
        })
        continue

    results = status["results"]
    comparison_rows.append({
        "Requested Engine": label,
        "Actual Engine": results.get("engine", "unknown"),
        "Best Model": results.get("best_model"),
        "Best Score": results.get("best_score"),
        "Task Type": results.get("task_type"),
        "Model ID": status.get("model_id"),
        "Error": None,
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Requested Engine")
print("--- Engine Comparison: Adult Census Income ---")
display(comparison_df)


In [ ]:
plot_df = comparison_df.dropna(subset=["Best Score"])
if not plot_df.empty:
    ax = plot_df["Best Score"].plot(kind="bar", color="#4C72B0", figsize=(8, 5))
    ax.set_title("Best Score by AutoML Engine — Adult Census Income")
    ax.set_ylabel("Best Score (AUC/accuracy, per engine's own metric)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No successful engine runs to plot -- check the 'Error' column above.")


## 9. Inspect a single engine's full leaderboard

`all_results` holds every candidate model each engine tried (not just the winner), so you can see how close the runner-up models were, or (for H2O/AutoGluon) inspect the underlying stacked-ensemble leaderboard.


In [ ]:
# Change this to "H2O" or "AutoGluon" to inspect a different engine's leaderboard
inspect_label = "FLAML"

inspect_status = engine_statuses.get(inspect_label)
if inspect_status and inspect_status.get("status") == "completed":
    all_results = inspect_status["results"]["all_results"]
    leaderboard_df = pd.DataFrame(all_results).T
    print(f"--- {inspect_label} full leaderboard ---")
    display(leaderboard_df)
else:
    print(f"{inspect_label} run did not complete successfully; nothing to show.")


## 10. Cleanup (optional)

The trained models stay registered in the backend's in-memory/model-dir store like any other Decisera model (usable via `/models/{model_id}/metrics`, `/models/predict`, etc.) until you remove them. `DELETE /api/v1/models/{model_id}` handles that properly: it drops the in-memory record, deletes the `.joblib` file, and (for the H2O run) also purges that run's full leaderboard from the H2O cluster's in-memory DKV store -- H2O keeps every candidate model resident, not just the leader, so skipping this step leaks JVM memory across repeated benchmark runs.


In [ ]:
delete_after_run = False  # flip to True to remove the three benchmark models this notebook just created

if delete_after_run:
    for label, status in engine_statuses.items():
        model_id = status.get("model_id")
        if not model_id:
            continue
        resp = requests.delete(f"{API_BASE}/models/{model_id}", timeout=30)
        resp.raise_for_status()
        print(f"[{label}] deleted model_id={model_id}: {resp.json()['message']}")
else:
    print("Skipping cleanup (delete_after_run=False). Set it to True to delete the benchmark models above.")
